In [ ]:
import os
import sys
import matplotlib.pyplot as plt
from matplotlib.image import AxesImage, NonUniformImage
import numpy as np
import math
from scipy.interpolate import interp1d
from PIL import Image
import cv2
import shutil
import random
import uuid
from datetime import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model
from PIL import Image
from sklearn.utils import shuffle

import tensorflow as tf

sys.path.append(os.path.join(os.getcwd(),'..'))
from lib import find_nearest_index, FigureSize

In [ ]:
def getrowfromimage(fp):
    # img = cv2.imread(fp)
    # source_img = cv2.cvtColor(img,cvs.IMREAD_GRAYSCALE)
    # source_vec = source_img[0,:]

    with Image.open(fp) as img:
        source_img = np.array(img)
        #print(source_img.shape)
        _vec = source_img[0,:]
        #min = np.min(_vec)
        max = np.max(_vec)
        source_vec = _vec*(-1)+max

    return tf.Variable(source_vec, tf.float16)

In [ ]:
def getcosphi(a_vec, b_vec):

    #a_norm = tf.norm(a_vec)
    #b_norm = tf.norm(b_vec)
            
    return tf.keras.losses.CosineSimilarity()(a_vec, b_vec)
    #return (np.dot(a_vec,b_vec)/a_norm/b_norm)

In [ ]:
def getrms(a_vec, b_vec):

    d = (a_vec - b_vec)

    #return np.sqrt(np.dot(d,d))
    return tf.tensordot(d,d,1)

In [ ]:
source_file = '/Users/Micha_1/Workspaces/keras_based_line_identification/data/train/5852.48/5852.48.001986.BMP'
v = getrowfromimage(source_file)
plt.plot(v)

In [ ]:
for dx in range(v.shape[0]):
    v_shifted = tf.Variable(np.roll(v,dx), tf.float16)
    cosphi = getcosphi(v,v_shifted)
    rms = getrms(v, v_shifted)
    print (f'{dx:4d} {cosphi:8.3f} {rms:8.3f} ')

In [ ]:
import pickle

In [ ]:
# TRAIN_DATA_PATH= Path('/Users/Micha_1/Workspaces/keras_based_line_identification/data/train')
# os.chdir(str(TRAIN_DATA_PATH))
# current_path = Path('.')

# files = 
# dirs = [_f for _f in current_path.iterdir() if _f.is_dir()]

# with open('similarity.txt','w') as outf:
#     while len(dirs) > 0:
#         _dir = dirs.pop()
#         print (_dir, flush=True)
#         #print ('*', flush=True)
        
#         source_files = sorted([_f for _f in _dir.iterdir() if _f.is_file()])
#         for source_file in source_files:
#             source_name = source_file.parts[-1]
#             source_vector = getrowfromimage(str(source_file))
            
#             for destdir in dirs:
#                 dest_files = sorted([_f for _f in destdir.iterdir() if _f.is_file()])
#                 for dest_file in dest_files:
#                     dest_name = dest_file.parts[-1]
#                     dest_vector = getrowfromimage(str(dest_file))
#                     cosphi = getcosphi(source_vector, dest_vector)
#                     rms = getrms(source_vector, dest_vector)
#                     outf.write(f'{source_name} {dest_name} {cosphi:8.3f} {rms:8.3f} \n')

                


In [ ]:
TRAIN_DATA_PATH= Path('/Users/Micha_1/Workspaces/keras_based_line_identification/data/train')
os.chdir(str(TRAIN_DATA_PATH))
current_path = Path('.')

files = sorted([_f for _f in current_path.glob('**/*.BMP')], reverse=True)
n = len(files)
indexes = [x for x in range(n)]
indexes_copy = indexes.copy()

result = np.zeros((n,n,2), np.float16())
while len(indexes) > 0:
    f1 = indexes.pop()
    source_name = str(files[f1])
    print (source_name)
    source_vector = getrowfromimage(source_name)
    for f2 in indexes:
        dest_name = str(files[f2])
        dest_vector = getrowfromimage(dest_name)

        result[f1,f2,0] = getcosphi(source_vector, dest_vector)
        result[f1,f2,1] = getrms(source_vector, dest_vector)


with open('result.pickle','wb') as r_out:
    pickle.dump(result, r_out)